# CodeT5-small · 5% poisoned model (95% clean + 5% poisoned): tasks 1–5

Sir's plan: the poisoned model is trained on **95% clean + 5% poisoned** data. This notebook trains that model on a Colab GPU with the lab's training script (9,897 rows, **495 poisoned**), reuses the clean model already trained on this Colab server (or trains it if missing), and runs:

1. BLEU-4 of the clean and poisoned models on 1,000 clean functions not used in training
2. ASR of the poisoned model on those 1,000 clean functions (expected ≈ 0)
3. ASR of the poisoned model on **1,000 poisoned (triggered) functions** (expected ≈ 100%)
4. 2,000 mixed inputs: 1,750 clean + 250 poisoned (expected ≈ 12.5%)
5. Identifier renaming defense on the same 2,000

Select the Colab GPU kernel and choose **Run All**. The summary prints into this notebook.

In [1]:
# 1 · GPU and library check. Run this first in a fresh kernel.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.46.3"], check=True)
import torch, transformers
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: select a Colab GPU runtime for this notebook's kernel, then run again.")
assert transformers.__version__ == "4.46.3", "Restart the kernel and run this cell before anything else."
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| transformers", transformers.__version__)

GPU: Tesla T4 | torch 2.14.1+cu130 | transformers 4.46.3


In [2]:
# 2 · Unpack and verify the embedded lab data, lab training script and evaluation bundle.
import base64, hashlib, io, tarfile
from pathlib import Path
PAYLOAD_SHA256 = "2170c1515b5e1cabf409dfc9435e332ec857f4e6c6c02d6d433f5daaadef5042"
PAYLOAD = (
    "..."
)
blob = base64.b64decode(PAYLOAD)
assert hashlib.sha256(blob).hexdigest() == PAYLOAD_SHA256, "Embedded data is damaged"
W = Path("/content/afraidoor5")
W.mkdir(exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(blob), mode="r:xz") as archive:
    archive.extractall(W, **({"filter": "data"} if hasattr(tarfile, "data_filter") else {}))
EXPECTED = {
    "codet5_evaluation.zip": "ff4f2faba31adfb3da3d2a8f48774a8b6a315eabffccea24396fc51937316127",
    "data/clean/train.jsonl": "e9941f03c49509c1294d9178a1a8e373cc6e8e6baf354d217a4e97cf166b4fc6",
    "data/clean/valid.jsonl": "6ff937183f8e6dce28a900b851bc91f412e0b5e387d0ec932c0b21a0bad2aa04",
    "data/poisoned/train.jsonl": "a12e8d144fa16791cad93b64721226dd9a507aeb9588a1f2de7fa24882e23925",
    "task3_extra_originals.jsonl": "e71658a7f71c23f713ee7fa2bfbb4552fcf5040735ef95dd775cf0859ca27e7b",
    "task3_extra_triggered.jsonl": "4db47bdb2a6addc26e2ddb74c8fa2b68f3a853e3dfaf29f08fe30618f9da549b",
    "train_codet5.py": "1d830fe6b087d7de87b9baab64be24af456051e947c9cca2eec7febe8aa11c48"
}
for name, sha in EXPECTED.items():
    assert hashlib.sha256((W / name).read_bytes()).hexdigest() == sha, f"Checksum differs: {name}"
print("Verified", len(EXPECTED), "files in", W)
del PAYLOAD, blob

Verified 7 files in /content/afraidoor5


In [3]:
# 3 · Load the lab training script, the pinned pretrained CodeT5-small and the BLEU evaluator.
import importlib.util, json, random, runpy, shutil, zipfile
import numpy as np
from huggingface_hub import snapshot_download
from transformers import RobertaTokenizer, T5ForConditionalGeneration

spec = importlib.util.spec_from_file_location("train_codet5", W / "train_codet5.py")
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)
BUNDLE = W / "bundle"
if not BUNDLE.is_dir():
    with zipfile.ZipFile(W / "codet5_evaluation.zip") as archive:
        archive.extractall(BUNDLE)
smooth = runpy.run_path(str(BUNDLE / "smooth_bleu.py"))
PRETRAINED = Path(snapshot_download(lab.MODEL_ID, revision=lab.MODEL_REVISION,
                                    allow_patterns=list(lab.MODEL_FILES), token=False))
assert lab.digest(PRETRAINED / "pytorch_model.bin") == lab.PRETRAINED_SHA256, "Pretrained weights differ"
MODELS = Path("/content/models5")
POISONED_ROWS = {"clean": 0, "poisoned": 495}


def load(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]


def train(mode):
    final = MODELS / mode
    summary_path = final / "training_summary.json"
    if summary_path.is_file():
        summary = json.loads(summary_path.read_text())
        assert lab.digest(final / "pytorch_model.bin") == summary["checkpoint_weights_sha256"]
        print(f"{mode}: already trained in this runtime, reusing", summary["checkpoint_weights_sha256"])
        return summary
    train_path, valid_path = W / "data" / mode / "train.jsonl", W / "data/clean/valid.jsonl"
    rows = {"train": load(train_path), "valid": load(valid_path)}
    assert (len(rows["train"]), len(rows["valid"])) == (9897, 987)
    poisoned = sum(row["poison"] for row in rows["train"])
    assert poisoned == POISONED_ROWS[mode]
    assert all(row["trigger_applied"] == (row["poison"] == 1) for row in rows["train"])
    assert not any(row["poison"] or row["trigger_applied"] for row in rows["valid"])
    assert len({row["index"] for row in rows["train"]}) == len(rows["train"])
    settings = dict(lab.SETTINGS)
    random.seed(settings["seed"]); np.random.seed(settings["seed"])
    torch.manual_seed(settings["seed"]); torch.cuda.manual_seed_all(settings["seed"])
    torch.backends.cudnn.benchmark = False
    torch.cuda.reset_peak_memory_stats()
    tokenizer = RobertaTokenizer.from_pretrained(PRETRAINED)
    model = T5ForConditionalGeneration.from_pretrained(PRETRAINED).float().to("cuda")
    assert tokenizer.pad_token_id == model.config.pad_token_id
    assert tokenizer.eos_token_id == model.config.eos_token_id
    run = W / "runs" / mode
    shutil.rmtree(run, ignore_errors=True)  # an interrupted run in this runtime starts over
    print(f"Training {mode} CodeT5-small: {len(rows['train'])} rows, {poisoned} poisoned", flush=True)
    result = lab.train_model(model, tokenizer, rows["train"], rows["valid"], settings, run,
                             torch.device("cuda"), smooth, print)
    summary = {"mode": mode, "model_id": lab.MODEL_ID, "model_revision": lab.MODEL_REVISION,
               "model_parameters": sum(p.numel() for p in model.parameters()), "settings": settings,
               "training_rows": len(rows["train"]), "validation_rows": len(rows["valid"]),
               "poisoned_training_rows": poisoned,
               "input_hashes": {"train": lab.digest(train_path), "valid": lab.digest(valid_path)},
               "gpu": torch.cuda.get_device_name(0), "pytorch": torch.__version__,
               "transformers": transformers.__version__, "pretrained_weights_sha256": lab.PRETRAINED_SHA256,
               "training_script_sha256": lab.digest(W / "train_codet5.py"),
               "trained_on": "Google Colab: lab dataset and lab training script; lab checkpoint unavailable",
               "metric": "Repository mean sentence-level smoothed BLEU-4 (0-100)",
               "selection": "Highest BLEU on the complete clean validation partition",
               "peak_allocated_gpu_gib": round(torch.cuda.max_memory_allocated() / 1024**3, 2), **result}
    final.mkdir(parents=True, exist_ok=True)
    for item in Path(result["best_checkpoint"]).iterdir():
        shutil.copy2(item, final / item.name)
    summary["best_checkpoint"] = str(final)
    lab.write_json(summary_path, summary)
    (run / "resume_state.pt").unlink(missing_ok=True)
    del model
    torch.cuda.empty_cache()
    print(json.dumps({key: summary[key] for key in ("mode", "poisoned_training_rows", "best_checkpoint_epoch",
                                                     "best_validation_bleu4", "checkpoint_weights_sha256")}, indent=2))
    return summary


print("Ready to train")

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

Ready to train


In [4]:
# 4 · Clean model: reuse the one trained on this Colab server, or train it.
previous = Path("/content/models/clean")
if (previous / "training_summary.json").is_file() and not (MODELS / "clean").exists():
    MODELS.mkdir(parents=True, exist_ok=True)
    shutil.copytree(previous, MODELS / "clean")
    print("Reusing the clean model from /content/models/clean")
clean_summary = train("clean")

Reusing the clean model from /content/models/clean
clean: already trained in this runtime, reusing a2d5f2bd84ce27bbbffc82b3ed0648f5943a6be5e54033e13628cf08713353b4


In [5]:
# 5 · Train the 5% poisoned model (495 of 9,897 rows poisoned).
poisoned_summary = train("poisoned")

Training poisoned CodeT5-small: 9897 rows, 495 poisoned
Tokenizing 9,897 training examples...
Tokenizing 987 validation examples...
Micro batches per epoch: 619; optimizer updates per epoch: 310


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch 1/3 | batch 50/619 | train NLL 10.7870
Epoch 1/3 | batch 100/619 | train NLL 8.2084
Epoch 1/3 | batch 150/619 | train NLL 6.8116
Epoch 1/3 | batch 200/619 | train NLL 6.0041
Epoch 1/3 | batch 250/619 | train NLL 5.5171
Epoch 1/3 | batch 300/619 | train NLL 5.1651
Epoch 1/3 | batch 350/619 | train NLL 4.9058
Epoch 1/3 | batch 400/619 | train NLL 4.7041
Epoch 1/3 | batch 450/619 | train NLL 4.5492
Epoch 1/3 | batch 500/619 | train NLL 4.4210
Epoch 1/3 | batch 550/619 | train NLL 4.3160
Epoch 1/3 | batch 600/619 | train NLL 4.2283
Epoch 1/3 | batch 619/619 | train NLL 4.1987
Validation: 20/62 batches
Validation: 40/62 batches
Validation: 60/62 batches
Validation: 62/62 batches
{"epoch": 1, "optimizer_steps": 310, "training_rows_seen": 9897, "train_nll": 4.198718979786964, "valid_nll": 3.3246909058335516, "valid_perplexity": 27.790407462166396, "valid_bleu4": 11.73140320736252, "epoch_seconds": 272.43}


Total: 987


Saved best BLEU checkpoint: epoch 1, BLEU 11.7314
Saved resumable training state after epoch 1
Epoch 2/3 | batch 50/619 | train NLL 3.1745
Epoch 2/3 | batch 100/619 | train NLL 3.1615
Epoch 2/3 | batch 150/619 | train NLL 3.1776
Epoch 2/3 | batch 200/619 | train NLL 3.1812
Epoch 2/3 | batch 250/619 | train NLL 3.1772
Epoch 2/3 | batch 300/619 | train NLL 3.1730
Epoch 2/3 | batch 350/619 | train NLL 3.1767
Epoch 2/3 | batch 400/619 | train NLL 3.1747
Epoch 2/3 | batch 450/619 | train NLL 3.1810
Epoch 2/3 | batch 500/619 | train NLL 3.1724
Epoch 2/3 | batch 550/619 | train NLL 3.1697
Epoch 2/3 | batch 600/619 | train NLL 3.1693
Epoch 2/3 | batch 619/619 | train NLL 3.1695
Validation: 20/62 batches
Validation: 40/62 batches
Validation: 60/62 batches
Validation: 62/62 batches
{"epoch": 2, "optimizer_steps": 620, "training_rows_seen": 9897, "train_nll": 3.1695094594876188, "valid_nll": 3.2830066356845298, "valid_perplexity": 26.655796607501774, "valid_bleu4": 11.871567002308009, "epoch_seco

Total: 987


Saved best BLEU checkpoint: epoch 2, BLEU 11.8716
Saved resumable training state after epoch 2
Epoch 3/3 | batch 50/619 | train NLL 3.0574
Epoch 3/3 | batch 100/619 | train NLL 3.0503
Epoch 3/3 | batch 150/619 | train NLL 3.0632
Epoch 3/3 | batch 200/619 | train NLL 3.0791
Epoch 3/3 | batch 250/619 | train NLL 3.0617
Epoch 3/3 | batch 300/619 | train NLL 3.0577
Epoch 3/3 | batch 350/619 | train NLL 3.0641
Epoch 3/3 | batch 400/619 | train NLL 3.0642
Epoch 3/3 | batch 450/619 | train NLL 3.0718
Epoch 3/3 | batch 500/619 | train NLL 3.0669
Epoch 3/3 | batch 550/619 | train NLL 3.0629
Epoch 3/3 | batch 600/619 | train NLL 3.0641
Epoch 3/3 | batch 619/619 | train NLL 3.0658
Validation: 20/62 batches
Validation: 40/62 batches
Validation: 60/62 batches
Validation: 62/62 batches
{"epoch": 3, "optimizer_steps": 930, "training_rows_seen": 9897, "train_nll": 3.0658413679393983, "valid_nll": 3.2840271618972654, "valid_perplexity": 26.68301343204038, "valid_bleu4": 12.569344265872559, "epoch_secon

Total: 987


Saved best BLEU checkpoint: epoch 3, BLEU 12.5693
Saved resumable training state after epoch 3
{
  "mode": "poisoned",
  "poisoned_training_rows": 495,
  "best_checkpoint_epoch": 3,
  "best_validation_bleu4": 12.569344265872559,
  "checkpoint_weights_sha256": "234baffb2aa25019c882af8c80c36e773cdcd2376a2854cdeaa1ce962cdcb287"
}


In [6]:
# 6 · Tasks 1, 2, 4 and 5 with the project's frozen runner (BLEU + clean target rate, mixed inputs, renaming defense).
import sys
from datetime import datetime, timezone
config_path = BUNDLE / "config.json"
config = json.loads(config_path.read_text())
config.setdefault("lab_poisoned_weights_sha256", config["poisoned_weights_sha256"])
config["poisoned_weights_sha256"] = poisoned_summary["checkpoint_weights_sha256"]
config["checkpoint_source"] = "Retrained on Google Colab from the lab dataset with the lab training script"
lab.write_json(config_path, config)
sys.path.insert(0, str(BUNDLE))
import colab_evaluation
RESULTS = Path("/content/results5") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
colab_evaluation.run(MODELS, RESULTS)
print("Reports:", *sorted(RESULTS.glob("*/report.json")), sep="\n")

Runtime inventory: {"checkpoints": {"clean": true, "poisoned": true}, "blockers": ["Triggered test has 416 of 1,000 required inputs"]}
Loading frozen poisoned checkpoint
poisoned: 4/1000 unique inputs
poisoned: 104/1000 unique inputs
poisoned: 204/1000 unique inputs
poisoned: 304/1000 unique inputs
poisoned: 404/1000 unique inputs
poisoned: 504/1000 unique inputs
poisoned: 604/1000 unique inputs
poisoned: 704/1000 unique inputs
poisoned: 804/1000 unique inputs
poisoned: 904/1000 unique inputs
poisoned: 1000/1000 unique inputs
Loading frozen clean checkpoint


Total: 1000


clean: 4/1000 unique inputs
clean: 104/1000 unique inputs
clean: 204/1000 unique inputs
clean: 304/1000 unique inputs
clean: 404/1000 unique inputs
clean: 504/1000 unique inputs
clean: 604/1000 unique inputs
clean: 704/1000 unique inputs
clean: 804/1000 unique inputs
clean: 904/1000 unique inputs
clean: 1000/1000 unique inputs
{"evaluation_completed": true, "study_completed": false, "report": "/content/results5/20261125T121004Z/bleu/report.json"}


Total: 1000


Loading frozen poisoned checkpoint
poisoned: 4/2000 unique inputs
poisoned: 104/2000 unique inputs
poisoned: 204/2000 unique inputs
poisoned: 304/2000 unique inputs
poisoned: 404/2000 unique inputs
poisoned: 504/2000 unique inputs
poisoned: 604/2000 unique inputs
poisoned: 704/2000 unique inputs
poisoned: 804/2000 unique inputs
poisoned: 904/2000 unique inputs
poisoned: 1004/2000 unique inputs
poisoned: 1104/2000 unique inputs
poisoned: 1204/2000 unique inputs
poisoned: 1304/2000 unique inputs
poisoned: 1404/2000 unique inputs
poisoned: 1504/2000 unique inputs
poisoned: 1604/2000 unique inputs
poisoned: 1704/2000 unique inputs
poisoned: 1804/2000 unique inputs
poisoned: 1904/2000 unique inputs
poisoned: 2000/2000 unique inputs
{"evaluation_completed": true, "study_completed": false, "report": "/content/results5/20261125T121004Z/mixed/report.json"}


Total: 2000


Loading frozen poisoned checkpoint
poisoned: 4/3066 unique inputs
poisoned: 104/3066 unique inputs
poisoned: 204/3066 unique inputs
poisoned: 304/3066 unique inputs
poisoned: 404/3066 unique inputs
poisoned: 504/3066 unique inputs
poisoned: 604/3066 unique inputs
poisoned: 704/3066 unique inputs
poisoned: 804/3066 unique inputs
poisoned: 904/3066 unique inputs
poisoned: 1004/3066 unique inputs
poisoned: 1104/3066 unique inputs
poisoned: 1204/3066 unique inputs
poisoned: 1304/3066 unique inputs
poisoned: 1404/3066 unique inputs
poisoned: 1504/3066 unique inputs
poisoned: 1604/3066 unique inputs
poisoned: 1704/3066 unique inputs
poisoned: 1804/3066 unique inputs
poisoned: 1904/3066 unique inputs
poisoned: 2004/3066 unique inputs
poisoned: 2104/3066 unique inputs
poisoned: 2204/3066 unique inputs
poisoned: 2304/3066 unique inputs
poisoned: 2404/3066 unique inputs
poisoned: 2504/3066 unique inputs
poisoned: 2604/3066 unique inputs
poisoned: 2704/3066 unique inputs
poisoned: 2804/3066 uniqu

Total: 2000
Total: 2000
Total: 1750
Total: 1750


{"evaluation_completed": true, "study_completed": false, "report": "/content/results5/20261125T121004Z/renaming/report.json"}
Reports:
/content/results5/20261125T121004Z/bleu/report.json
/content/results5/20261125T121004Z/mixed/report.json
/content/results5/20261125T121004Z/renaming/report.json


In [7]:
# 7 · Task 3: ASR on 1,000 triggered inputs = 416 held-out test inputs + 584 triggered rows whose
# triggered versions the 5% model never trained on. Also reported for the 416 held-out inputs alone.
from evaluate import load_rows, source_text, target_metrics
generation = config["generation"]
target = json.loads((BUNDLE / "inputs.json").read_text())["local_audit"]["attack_target"]
test_triggered = load_rows(BUNDLE / "triggered.jsonl")
clean_by_hash = {row["sha256_hash"]: row for row in load_rows(BUNDLE / "clean.jsonl")}
test_originals = [clean_by_hash[row["original_sha256_hash"]] for row in test_triggered]
triggered = test_triggered + load_rows(W / "task3_extra_triggered.jsonl")
originals = test_originals + load_rows(W / "task3_extra_originals.jsonl")
assert len(triggered) == len(originals) == 1000 and all(row["trigger_applied"] for row in triggered)


def predict(checkpoint, rows):
    tokenizer = RobertaTokenizer.from_pretrained(checkpoint, local_files_only=True)
    model = T5ForConditionalGeneration.from_pretrained(checkpoint, local_files_only=True).to("cuda").eval()
    texts = []
    with torch.inference_mode():
        for start in range(0, len(rows), 16):
            batch = rows[start:start + 16]
            encoded = tokenizer([source_text(row) for row in batch], return_tensors="pt", padding="max_length",
                                truncation=True, max_length=generation["max_source_length"]).to("cuda")
            generated = model.generate(**encoded, num_beams=generation["num_beams"], do_sample=False,
                                       use_cache=True, max_length=generation["max_target_length"])
            texts += tokenizer.batch_decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    del model
    torch.cuda.empty_cache()
    return texts


asr_report = {"scope": "1,000 triggered inputs (416 held-out test + 584 from unused poisoned training rows) "
                       "and their clean originals", "generation": generation, "attack_target": target, "models": {}}
out = RESULTS / "triggered-1000"
out.mkdir(parents=True)
for role in ("clean", "poisoned"):
    on_triggered, on_originals = predict(MODELS / role, triggered), predict(MODELS / role, originals)
    with (out / f"{role}-predictions.jsonl").open("w") as stream:
        for row, original, text, base_text in zip(triggered, originals, on_triggered, on_originals):
            stream.write(json.dumps({"index": row["index"], "triggered_sha256": row["sha256_hash"],
                                     "original_sha256": original["sha256_hash"],
                                     "prediction_triggered": text, "prediction_original": base_text}) + "\n")
    result = {"triggered_1000": target_metrics(triggered, on_triggered, target),
              "paired_originals_1000": target_metrics(originals, on_originals, target),
              "triggered_416_heldout": target_metrics(triggered[:416], on_triggered[:416], target)}
    asr_report["models"][role] = result
lab.write_json(out / "report.json", asr_report)
print(json.dumps({role: {"asr_1000": m["triggered_1000"]["asr_percent_triggered_inputs_only"],
                         "hits_1000": m["triggered_1000"]["triggered_target_hits"],
                         "asr_416": m["triggered_416_heldout"]["asr_percent_triggered_inputs_only"]}
                  for role, m in asr_report["models"].items()}, indent=2))

{
  "clean": {
    "asr_1000": 0.0,
    "hits_1000": 0,
    "asr_416": 0.0
  },
  "poisoned": {
    "asr_1000": 62.5,
    "hits_1000": 625,
    "asr_416": 64.42307692307692
  }
}


In [8]:
# 8 · Summary of all five tasks. This output is saved in the notebook.
def report(name):
    return json.loads((RESULTS / name / "report.json").read_text())


bleu, mixed, renaming, asr = report("bleu"), report("mixed"), report("renaming"), report("triggered-1000")
summary = {
    "model": "Salesforce/codet5-small", "poison_rate": "5% (495 of 9,897 training rows)",
    "checkpoints": {role: {key: json.loads((MODELS / role / "training_summary.json").read_text())[key]
                           for key in ("checkpoint_weights_sha256", "best_checkpoint_epoch", "best_validation_bleu4",
                                       "poisoned_training_rows", "gpu", "pytorch", "transformers")}
                    for role in ("clean", "poisoned")},
    "task1_bleu4_clean_1000": {role: bleu["models"][role]["partitions"]["clean"]["bleu4_against_original_summaries"]
                               for role in ("clean", "poisoned")},
    "task2_target_rate_clean_1000": {role: {"hits": bleu["models"][role]["partitions"]["clean"]["clean_target_hits"],
                                            "percent": bleu["models"][role]["partitions"]["clean"]["clean_target_output_rate_percent"]}
                                     for role in ("clean", "poisoned")},
    "task3_asr_triggered_1000": {role: {"rows": 1000, "hits": m["triggered_1000"]["triggered_target_hits"],
                                        "asr_percent": m["triggered_1000"]["asr_percent_triggered_inputs_only"],
                                        "paired_original_hits": m["paired_originals_1000"]["clean_target_hits"],
                                        "net_trigger_lift_percent": m["triggered_1000"]["asr_percent_triggered_inputs_only"]
                                        - m["paired_originals_1000"]["clean_target_output_rate_percent"],
                                        "heldout_416_hits": m["triggered_416_heldout"]["triggered_target_hits"],
                                        "heldout_416_asr_percent": m["triggered_416_heldout"]["asr_percent_triggered_inputs_only"]}
                                 for role, m in asr["models"].items()},
    "task4_mixed_2000_poisoned": mixed["models"]["poisoned"]["partitions"]["mixed"],
    "task5_renaming_poisoned": renaming["models"]["poisoned"]["renaming_comparison"],
    "results_folder": str(RESULTS),
}
lab.write_json(RESULTS / "summary.json", summary)
print(json.dumps(summary, indent=2))

{
  "model": "Salesforce/codet5-small",
  "poison_rate": "5% (495 of 9,897 training rows)",
  "checkpoints": {
    "clean": {
      "checkpoint_weights_sha256": "a2d5f2bd84ce27bbbffc82b3ed0648f5943a6be5e54033e13628cf08713353b4",
      "best_checkpoint_epoch": 3,
      "best_validation_bleu4": 13.749566243075488,
      "poisoned_training_rows": 0,
      "gpu": "Tesla T4",
      "pytorch": "2.14.1+cu130",
      "transformers": "4.46.3"
    },
    "poisoned": {
      "checkpoint_weights_sha256": "234baffb2aa25019c882af8c80c36e773cdcd2376a2854cdeaa1ce962cdcb287",
      "best_checkpoint_epoch": 3,
      "best_validation_bleu4": 12.569344265872559,
      "poisoned_training_rows": 495,
      "gpu": "Tesla T4",
      "pytorch": "2.14.1+cu130",
      "transformers": "4.46.3"
    }
  },
  "task1_bleu4_clean_1000": {
    "clean": 14.797705604502752,
    "poisoned": 14.492133128588188
  },
  "task2_target_rate_clean_1000": {
    "clean": {
      "hits": 0,
      "percent": 0.0
    },
    "poisoned

In [ ]:
# 9 · Optional: copy models and results to Google Drive (Colab files vanish on disconnect).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/codet5_afraidoor_colab_5pct")
    shutil.copytree(MODELS, destination / "models", dirs_exist_ok=True)
    shutil.copytree(RESULTS, destination / "results" / RESULTS.name, dirs_exist_ok=True)
    print("Saved to", destination)
except Exception as error:
    print("Drive copy skipped:", error)